In [1]:
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/content/drive/MyDrive/Monetary_Economics_Feature_Engineering.csv')
df.head()

,Country,Date,CPI Inflation,Exchange Rate,Lending Interest Rate,World Oil Price,Global Economic Activity,wop_lag1,wop_lag2,gea_lag1,wop_pct_change,oil_moving_average,gea_moving_average,wop_volatility,Global_Pressure_Index
0,ALGERIA,01/01/2000,2.242235,70.7292,10.0,27.224286,-10.028534,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.380702
1,ALGERIA,01/02/2000,2.426637,72.4357,10.0,29.362381,-8.681862,27.224286,NaN,-10.028534,0.078536,NaN,NaN,NaN,0.332437
2,ALGERIA,01/03/2000,1.660758,73.4741,10.0,29.892174,5.995629,29.362381,27.224286,-8.681862,0.018043,NaN,NaN,NaN,0.052577
3,ALGERIA,01/04/2000,0.924354,76.0025,10.0,25.799000,9.081429,29.892174,29.362381,5.995629,-0.136931,28.826280,-4.238256,1.412430,-0.005800
4,ALGERIA,01/05/2000,-0.485800,75.2502,10.0,28.833478,5.096124,25.799000,29.892174,9.081429,0.117620,28.351185,2.131732,2.226074,0.071619


In [3]:
df.columns

Index(['Country', 'Date', 'CPI Inflation', 'Exchange Rate',
       ' Lending Interest Rate', 'World Oil Price', 'Global Economic Activity',
       'wop_lag1', 'wop_lag2', 'gea_lag1', 'wop_pct_change',
       'oil_moving_average', 'gea_moving_average', 'wop_volatility',
       'Global_Pressure_Index'],
      dtype='object')

# The EXR Conversion to EXR_Log_Returns
This is to erase the non stationary effect of the data as the curve can rise or dip randomly(Random Walk Hypothesis)

# The Problem:
The "average" number is constantly changing depending on the year. In 2000, the average might be 50 KSh; in 2026, the average is 130 KSh. Because the numbers can climb out to infinity, it has no fixed anchor.

# Solution WorkFlow

[Raw Historical CSV: 130 KSh]
       │
       ▼ (np.log / shift)
       

[Stationary Shock: +0.015] ──► [Fed to LightGBM Backend for Training]
                                           │
                                           ▼ (Model Predicts Next Shock: -0.008)

[Streamlit Frontend App]  ◄── [Converted: Current Price * np.exp(-0.008)]
       │
       ▼
[UI Screen Displays: "128.96 KSh/USD"]

In [4]:
# EXR Conversion to only show Random Shocks
df['EXR_Log_Returns'] = df.groupby('Country')['Exchange Rate'].transform(
    lambda x: np.log(x).diff()
)

In [5]:
# Drop EXR(Raw Data)
df.drop('Exchange Rate', axis=1).reset_index(drop=True)

,Country,Date,CPI Inflation,Lending Interest Rate,World Oil Price,Global Economic Activity,wop_lag1,wop_lag2,gea_lag1,wop_pct_change,oil_moving_average,gea_moving_average,wop_volatility,Global_Pressure_Index,EXR_Log_Returns
0,ALGERIA,01/01/2000,2.242235,10.000000,27.224286,-10.028534,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.380702,NaN
1,ALGERIA,01/02/2000,2.426637,10.000000,29.362381,-8.681862,27.224286,NaN,-10.028534,0.078536,NaN,NaN,NaN,0.332437,0.023841
2,ALGERIA,01/03/2000,1.660758,10.000000,29.892174,5.995629,29.362381,27.224286,-8.681862,0.018043,NaN,NaN,NaN,0.052577,0.014234
3,ALGERIA,01/04/2000,0.924354,10.000000,25.799000,9.081429,29.892174,29.362381,5.995629,-0.136931,28.826280,-4.238256,1.412430,-0.005800,0.033833
4,ALGERIA,01/05/2000,-0.485800,10.000000,28.833478,5.096124,25.799000,29.892174,9.081429,0.117620,28.351185,2.131732,2.226074,0.071619,-0.009948
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2875,UGANDA,01/08/2023,3.463346,18.398591,81.372609,-39.480500,75.766667,70.306818,-49.203251,0.073990,72.582321,-37.564787,2.841124,-0.495595,0.028139
2876,UGANDA,01/09/2023,2.680781,18.954428,89.240952,-20.546293,81.372609,75.766667,-39.480500,0.096695,75.815365,-44.641925,5.533056,-0.431249,0.010434
2877,UGANDA,01/10/2023,2.434750,18.902427,85.469091,8.871201,89.240952,81.372609,-20.546293,-0.042266,82.126743,-36.410015,6.768725,0.000826,0.006227
2878,UGANDA,01/11/2023,2.605582,16.785686,77.575455,6.896983,85.469091,89.240952,8.871201,-0.092357,85.360884,-17.051864,3.935288,-0.015509,0.009604


# The Augmented Dickey Fuller Test (ADF Test)

The ADF test is a statistical test used to check whether a time-series variable is stationary or non-stationary.

# How the ADF Test Works Behind the Scenes

The ADF test works like a court case. It sets up two opposing arguments, known as hypotheses:

## The Null & Alternative Hypothesis

**The Null Hypothesis ($H_0$)**: The data is non-stationary. It possesses a "unit root," meaning it behaves like a drifting random walk with a changing mean or exploding variance over time. This is the dangerous state.

**The Alternative Hypothesis ($H_1$)**: The data is stationary. It has no trend, a constant mean, and stable variance. This is the green-light state.

To determine who wins the court case, the mathematical test analyzes your target variable's history and outputs a single crucial number: the p-value.

## The Decision Rules:

**If p-value $\ge$ 0.05**: You fail to reject the null hypothesis. The data is officially non-stationary (unsafe).

**If p-value $<$ 0.05**: You successfully reject the null hypothesis! The court rules that your data is stationary (safe).

In [6]:
# Augmented Dickey Fuller Test
variables = ['CPI Inflation', ' Lending Interest Rate', 'EXR_Log_Returns']
for var in variables:
    clean_series = df[var].dropna()
    result = adfuller(clean_series)
    p_value = result[1]

    print(f"Variable: {var:<15} | ADF Stat: {result[0]:.4f} | p-value: {p_value:.4f}")
    if p_value <= 0.05:
        print("Conclusion: STATIONARY \n")
    else:
        print("Conclusion: NON-STATIONARY (Requires transformation)\n")

Variable: CPI Inflation   | ADF Stat: -6.3224 | p-value: 0.0000
Conclusion: STATIONARY 

Variable:  Lending Interest Rate | ADF Stat: -5.8179 | p-value: 0.0000
Conclusion: STATIONARY 

Variable: EXR_Log_Returns | ADF Stat: -14.1149 | p-value: 0.0000
Conclusion: STATIONARY 



In [7]:
df.columns

Index(['Country', 'Date', 'CPI Inflation', 'Exchange Rate',
       ' Lending Interest Rate', 'World Oil Price', 'Global Economic Activity',
       'wop_lag1', 'wop_lag2', 'gea_lag1', 'wop_pct_change',
       'oil_moving_average', 'gea_moving_average', 'wop_volatility',
       'Global_Pressure_Index', 'EXR_Log_Returns'],
      dtype='object')

# MultiOutPut Architectural Model (How it was Solved)

Standard LightGBM, like most machine learning models, is designed to be a single-task engine. It takes in your features and can only output a single numerical prediction at a time (e.g., just predicting Inflation).

## The Scikit-Learn MultiOutput Regressor
This allows to output three variables :

* CPI Inflation

* Lending Interest Rate

* EXR Log Returns

as output to our streamlit dashboard simultaneously.

# Light Gradient Boosting Machine (LightGBM)

Our model's workflow begins by using a MultiOutputRegressor manager to clone a base LightGBM algorithm into three parallel, independent decision-tree tracks—one dedicated to each of your stationary targets (Inflation, Interest Rates, and Exchange Rate Log Returns).

For each track, LightGBM trains sequentially by building an ensemble of trees where each new tree is optimized via a leaf-wise (best-first) growth strategy to specifically correct the residual prediction errors of the trees that came before it.

This allows the model to quickly and accurately map your lag features and global stress indicators directly to the highly volatile percentage shocks of all three macroeconomic variables simultaneously.

In [8]:
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor

# Generation Of Lags On Target Variables

We generate lags for our target variables to give our tabular machine learning model a "memory" of time progression, allowing it to capture the natural momentum and delayed transmission effects of macroeconomics

In [9]:
# Generate Lags for your targets
targets = ['CPI Inflation', ' Lending Interest Rate', 'EXR_Log_Returns']

for col in targets:
    df[f'{col}_Lag1'] = df.groupby('Country')[col].shift(1)
    df[f'{col}_Lag2'] = df.groupby('Country')[col].shift(2)

# Feature & Test Splitting

In [10]:
# Final Feature space (X) and Target space (y)
X_features = [
    'wop_lag1', 'wop_lag2', 'gea_lag1', 'wop_pct_change',
    'oil_moving_average', 'gea_moving_average', 'wop_volatility', 'Global_Pressure_Index',
    'CPI Inflation_Lag1', 'CPI Inflation_Lag2',
    ' Lending Interest Rate_Lag1', ' Lending Interest Rate_Lag2',
    'EXR_Log_Returns_Lag1', 'EXR_Log_Returns_Lag2'
]

# Simultenous Prediction Targets
y_targets = ['CPI Inflation', ' Lending Interest Rate', 'EXR_Log_Returns']

# Categorical Variable Mapping and encoding

In [11]:
df['Country'] = df['Country'].astype('category')
encoder = OrdinalEncoder()

df['Country'] = encoder.fit_transform(df[['Country']])

X_features.append('Country')

# Model Training Phase

The Model will be trained on 100 percent of the dataset

### This is the Production Model


In [12]:
train_df = df.dropna(subset=X_features + y_targets)

X = train_df[X_features]
y = train_df[y_targets]

In [13]:
base_model = LGBMRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

# 3. Wrap and Fit
model = MultiOutputRegressor(base_model)
model.fit(X, y)

print("Multi-Output LightGBM Model successfully trained with encoded features! ")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004563 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3580
[LightGBM] [Info] Number of data points in the train set: 2850, number of used features: 15
[LightGBM] [Info] Start training from score 12.186481
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

# Model Evaluation Phase Using Time-Series BackTesting

In [14]:
# Converted Date and calculated the cut-off Date
df['Date'] = pd.to_datetime(df['Date'])
cut_off_date = df['Date'].max() - pd.DateOffset(months=12)

# I cleaned rows in order to create train_df
train_df = df.dropna(subset=X_features + y_targets)

# Create a training_for_backtest and backtest_data
training_for_backtest = train_df['Date'] < cut_off_date
backtest_data = train_df['Date'] >= cut_off_date

In [15]:
X_train, X_test = X[training_for_backtest], X[backtest_data]
y_train, y_test = y[training_for_backtest], y[backtest_data]

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000703 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3580
[LightGBM] [Info] Number of data points in the train set: 2720, number of used features: 15
[LightGBM] [Info] Start training from score 11.998465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

In [16]:
y_pred_df = pd.DataFrame(y_pred, columns=y_targets, index=y_test.index)

# Accuracy Metrics
print("--- MODEL EVALUATION METRICS ---")
for col in y_targets:
    actual = y_test[col]
    predicted = y_pred_df[col]

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    print(f"\n Target: {col}")
    print(f"   Mean Absolute Error (MAE): {mae:.4f}")
    print(f"   Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"   R² Score (Variance Explained): {r2:.4f}")

--- MODEL EVALUATION METRICS ---

 Target: CPI Inflation
   Mean Absolute Error (MAE): 1.4769
   Root Mean Squared Error (RMSE): 2.5110
   R² Score (Variance Explained): 0.9553

 Target:  Lending Interest Rate
   Mean Absolute Error (MAE): 0.7638
   Root Mean Squared Error (RMSE): 1.5212
   R² Score (Variance Explained): 0.9546

 Target: EXR_Log_Returns
   Mean Absolute Error (MAE): 0.0331
   Root Mean Squared Error (RMSE): 0.0803
   R² Score (Variance Explained): -0.0216


In [17]:
import joblib

# Save the trained model and encoder to disk for the Streamlit app to read
joblib.dump(model, 'multi_output_lgbm.pkl')
joblib.dump(encoder, 'country_encoder.pkl')
print("Models and Encoders saved successfully!")

Models and Encoders saved successfully!
